# 07 · Evaluation & Fairness

> **Synthetic data only.** Every dataset used in this notebook is synthetic (Synthea, seed 42) or research-use-only (NIH ChestX-ray14). There is no PHI. None of these models is clinically validated or cleared for patient care.

Evaluate a model with the shared `medflow_ml.evaluation.metrics` and produce the two artifacts every MedFlow model card needs: a **subgroup AUROC table** (sex / age band / race) and **calibration-by-group** curves. Synthetic scores throughout.

## Synthesize labeled predictions with subgroups
We build a reproducible synthetic evaluation frame: a label, a model score, and protected attributes for subgroup analysis.

In [ ]:
from __future__ import annotations
import numpy as np, pandas as pd
from medflow_ml.features.encounters import age_band
rng = np.random.default_rng(42)
N = 1000
age = rng.integers(20, 90, size=N)
sex = rng.choice(['female', 'male'], size=N)
race = rng.choice(['white', 'black', 'asian', 'other'], size=N)
base = 0.08 + 0.004 * (age - 50) + 0.05 * (sex == 'male')
y = (rng.random(N) < np.clip(base, 0.02, 0.9)).astype(int)
# A decent-but-imperfect scorer with mild sex-dependent noise.
noise = rng.normal(0, 0.15 + 0.05 * (sex == 'female'), size=N)
score = np.clip(0.5 * y + 0.5 * rng.random(N) + noise, 0, 1)
df = pd.DataFrame({'y': y, 'score': score, 'sex': sex, 'race': race,
                   'age_band': [age_band(int(a)) for a in age]})
df.head()

## Overall discrimination & calibration

In [ ]:
from medflow_ml.evaluation.metrics import (
    auroc, auprc, expected_calibration_error, calibration_curve,
)
print('AUROC:', round(auroc(df['y'], df['score']), 3))
print('AUPRC:', round(auprc(df['y'], df['score']), 3))
print('ECE  :', round(expected_calibration_error(df['y'], df['score']), 3))

## Subgroup AUROC table
The core fairness artifact: AUROC computed within each subgroup label. `subgroup_auroc` returns `nan` for single-class groups (undefined).

In [ ]:
from medflow_ml.evaluation.metrics import subgroup_auroc
def subgroup_table(frame, col):
    d = subgroup_auroc(frame['y'], frame['score'], frame[col])
    return pd.DataFrame({'subgroup': list(d), 'auroc': [round(v, 3) for v in d.values()]})
tables = {c: subgroup_table(df, c) for c in ['sex', 'age_band', 'race']}
pd.concat([t.assign(attribute=a) for a, t in tables.items()], ignore_index=True)

## Calibration-by-group
Reliability curves per sex. A model that is well calibrated overall can still be miscalibrated within a subgroup — this plot surfaces that.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect')
for g, sub in df.groupby('sex'):
    mean_pred, obs_freq, _ = calibration_curve(sub['y'], sub['score'], n_bins=8)
    ax.plot(mean_pred, obs_freq, marker='o', label=f'sex={g}')
ax.set_xlabel('mean predicted probability')
ax.set_ylabel('observed frequency')
ax.set_title('Calibration by group'); ax.legend(); plt.tight_layout()

## Operating points
Sensitivity/specificity/PPV/alert-rate at candidate thresholds, for clinically-meaningful threshold selection.

In [ ]:
from medflow_ml.evaluation.metrics import operating_point_metrics
pts = operating_point_metrics(df['y'], df['score'], thresholds=[0.3, 0.5, 0.7])
pd.DataFrame([p.__dict__ for p in pts]).round(3)

### Takeaways
* Always report **subgroup** AUROC and **calibration-by-group**, not just overall metrics.
* `nan` subgroups signal too few samples / single class — flag, don't hide.
* These tables feed directly into each model card's fairness section.